# Task 04: Supervised Classification / Regression Modeling & Tuning

**Project:** Customer Churn Prediction  
**Student:** Tharani P  
**Goal:** Compare four classification algorithms, tune hyperparameters with stratified cross-validation, evaluate performance, select a champion model, and save it.

## Before running
1. Upload `customer-churn-training.csv` from your Task 03 project into the notebook session.
2. Run all cells from top to bottom.
3. The notebook automatically recognises common target names, including `churned`. If needed, set `TARGET_COLUMN` in the configuration cell.
4. The supplied sample has only 12 rows, so all metrics are highly unstable and for classroom demonstration only.
5. Do not publish the CSV publicly unless you have permission to share the data.


## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import joblib

print("Libraries imported successfully.")

## 2. Load the dataset

In [ ]:
DATA_PATH = "customer-churn-training.csv"
TARGET_COLUMN = None  # Optional: enter exact target column name, e.g. "Churn"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Upload customer-churn-training.csv to the notebook session, "
        "or update DATA_PATH to the correct location."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumns:")
print(df.columns.tolist())

## 3. Identify the target and inspect data

In [ ]:
if TARGET_COLUMN is None:
    target_candidates = [
        c for c in df.columns
        if c.strip().lower() in {"churn", "churned", "exited", "target", "label", "class", "customer_churn", "customer_churned"}
    ]
    if not target_candidates:
        raise ValueError(
            "Target column was not detected automatically. Set TARGET_COLUMN in the previous cell "
            "to the exact name of your churn/target column."
        )
    TARGET = target_candidates[0]
else:
    TARGET = TARGET_COLUMN

print("Target column:", TARGET)
print("\\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_values"))
print("\\nTarget distribution:")
display(df[TARGET].value_counts(dropna=False).to_frame("count"))

df = df.dropna(subset=[TARGET]).copy()
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

# Remove common ID-like columns because they usually do not generalize as useful predictors.
id_like = [c for c in X.columns if c.strip().lower() in {"id", "customerid", "customer_id", "row_id"}]
if id_like:
    X = X.drop(columns=id_like)
    print("Dropped ID-like columns:", id_like)

# Encode target labels as integers for consistent classification metrics.
target_encoder = LabelEncoder()
y = pd.Series(target_encoder.fit_transform(y.astype(str)), index=y.index, name=TARGET)
print("Target classes:", list(target_encoder.classes_))
print("Feature matrix shape:", X.shape)

## 4. Split into training and test sets

In [ ]:
class_counts = y.value_counts()
if len(class_counts) < 2:
    raise ValueError("The target must contain at least two classes for classification.")

# Use stratification only when every class has at least two examples.
can_stratify = class_counts.min() >= 2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
    stratify=y if can_stratify else None
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training class counts:")
display(y_train.value_counts().to_frame("count"))

## 5. Build reusable preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## 6. Configure four classification models and hyperparameter grids

In [ ]:
models_and_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced"),
        {
            "model__C": [0.1, 1.0, 10.0],
            "model__solver": ["liblinear"]
        }
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42, class_weight="balanced"),
        {
            "model__max_depth": [None, 3, 5],
            "model__min_samples_split": [2, 4]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42, class_weight="balanced"),
        {
            "model__n_estimators": [50, 100],
            "model__max_depth": [None, 3, 5],
            "model__min_samples_split": [2, 4]
        }
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.03, 0.1],
            "model__max_depth": [1, 3]
        }
    )
}

print("Models configured:", list(models_and_grids.keys()))

## 7. Tune models using stratified cross-validation

In [ ]:
min_train_class_count = int(y_train.value_counts().min())
n_splits = min(5, min_train_class_count)

if n_splits < 2:
    raise ValueError(
        "There are too few examples per class in the training split for stratified cross-validation. "
        "Use a larger dataset or collect more examples for each class."
    )

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
scoring_metric = "f1_weighted"

fitted_searches = {}
comparison_rows = []

for model_name, (estimator, param_grid) in models_and_grids.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring_metric,
        cv=cv,
        n_jobs=-1,
        refit=True,
        error_score="raise"
    )
    search.fit(X_train, y_train)
    fitted_searches[model_name] = search

    best_model = search.best_estimator_
    predictions = best_model.predict(X_test)

    if hasattr(best_model, "predict_proba"):
        probabilities = best_model.predict_proba(X_test)
        if probabilities.shape[1] == 2:
            try:
                roc_auc = roc_auc_score(y_test, probabilities[:, 1])
            except ValueError:
                roc_auc = np.nan
        else:
            try:
                roc_auc = roc_auc_score(y_test, probabilities, multi_class="ovr", average="weighted")
            except ValueError:
                roc_auc = np.nan
    else:
        roc_auc = np.nan

    comparison_rows.append({
        "Model": model_name,
        "Best CV F1 (weighted)": search.best_score_,
        "Test Accuracy": accuracy_score(y_test, predictions),
        "Test Precision (weighted)": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "Test Recall (weighted)": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "Test F1 (weighted)": f1_score(y_test, predictions, average="weighted", zero_division=0),
        "Test ROC-AUC": roc_auc,
        "Best Parameters": search.best_params_
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by="Best CV F1 (weighted)", ascending=False
).reset_index(drop=True)

display(comparison_df)

## 8. Select the champion model

In [ ]:
champion_name = comparison_df.loc[0, "Model"]
champion_search = fitted_searches[champion_name]
champion_model = champion_search.best_estimator_

print("Champion model:", champion_name)
print("Best cross-validation F1 (weighted):", round(float(champion_search.best_score_), 4))
print("Best parameters:", champion_search.best_params_)

champion_predictions = champion_model.predict(X_test)
print("\\nClassification report:")
print(classification_report(y_test, champion_predictions, zero_division=0))

## 9. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, champion_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title(f"Confusion Matrix - {champion_name}")
plt.tight_layout()
plt.show()

## 10. ROC curve (binary classification)

In [ ]:
if len(np.unique(y_test)) == 2 and hasattr(champion_model, "predict_proba"):
    champion_probabilities = champion_model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, champion_probabilities)
    auc_value = roc_auc_score(y_test, champion_probabilities)

    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f"{champion_name} (AUC = {auc_value:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC-AUC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("ROC curve skipped: this cell requires a binary test set with both classes.")

## 11. Save the champion model

In [ ]:
MODEL_PATH = "best_churn_model.joblib"
joblib.dump({
    "model": champion_model,
    "target_encoder": target_encoder,
    "target_column": TARGET,
    "feature_columns": X.columns.tolist(),
    "champion_model_name": champion_name
}, MODEL_PATH)

print("Saved champion model to:", MODEL_PATH)

## 12. Conclusion

In [ ]:
print(f"Champion model selected: {champion_name}")
print("Selection basis: highest weighted F1-score from cross-validation.")
print("The saved model file is best_churn_model.joblib")
print("\\nImportant limitation:")
print("If the dataset contains only a small number of rows, the scores and selected model may be unstable.")
print("Use a larger, representative dataset before drawing real-world conclusions.")